<a href="https://colab.research.google.com/github/e3la/i2dc/blob/main/reels_srt_cpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to i2dc: Reels Review Tool - CPU (Slow) Edition
This is a tool for generating subtitles and reviewing metadata for your reels. It uses ffmpeg and whisper (on cpu) and is part of the i2dc toolkit find out more at https://github.com/e3la/i2dc/. If you can use GPU you probably should because it is faster.

In [ ]:
#@title <h1> **Step 1: Setup Environment**
# @markdown Run this cell once per session.

# --- Installations & Imports ---\n",
print("--- Setting up environment ---")
print("Installing required Python packages (Pandas, Pillow, OpenPyXL, Whisper) and ffmpeg...")
!pip install -q Pillow openpyxl pandas git+https://github.com/openai/whisper.git
!apt-get -qq install ffmpeg > /dev/null

import os
import shutil
import zipfile
import pandas as pd
from google.colab import files, drive
import glob
from IPython import get_ipython
from PIL import Image as PILImage, ImageDraw # Use alias to avoid name conflicts
import subprocess # Needed for running ffmpeg and whisper
print("✅ Environment is ready.")


# --- Define constants ---\n",
REVIEW_DIR = "/content/review_data"
METADATA_FILENAME_PATTERN = "*metadata*.xlsx"
GDRIVE_I2DC_PATH = "/content/drive/MyDrive/i2dc"
VIDEO_PLACEHOLDER_PATH = "/content/video_placeholder.png"

# --- Define Global Helper functions ---\n",
def create_video_placeholder(path, width=400, height=225):
    """Creates a generic video placeholder image."""
    img = PILImage.new('RGB', (width, height), color = '#E0E0E0') # Light grey background
    d = ImageDraw.Draw(img)
    triangle_size = 40
    center_x, center_y = width // 2, height // 2
    p1 = (center_x - triangle_size // 2, center_y - triangle_size // 2)
    p2 = (center_x - triangle_size // 2, center_y + triangle_size // 2)
    p3 = (center_x + triangle_size // 2, center_y)
    d.polygon([p1, p2, p3], fill = '#FFFFFF', outline = '#BDBDBD')
    img.save(path)

def generate_thumbnail(video_path, output_path):
    """Generates a thumbnail for a video file using ffmpeg."""
    try:
        command = ['ffmpeg', '-ss', '00:00:01.00', '-i', video_path, '-vframes', '1', '-q:v', '2', '-y', output_path]
        subprocess.run(command, check=True, capture_output=True, text=True)
        return True
    except (subprocess.CalledProcessError, FileNotFoundError):
        return False

print("\n---> You are now ready to proceed to Step 2 to load your data.")

In [ ]:
#@title <h1> **Step 2: Load ZIP Package & Pre-process Data**

# @markdown Run this cell to select and load your data. This tool will automatically filter for video files (.mp4, .mov, .webm) to prepare for the SRT generation workflow.

print("--- Resetting session for new data package ---")
# --- Reset variables and clean up previous data ---
zip_filepath = None
df = None
using_gdrive = False
if os.path.exists(REVIEW_DIR):
    shutil.rmtree(REVIEW_DIR)
os.makedirs(REVIEW_DIR, exist_ok=True)
create_video_placeholder(VIDEO_PLACEHOLDER_PATH) # Re-create the placeholder image

# --- Ask user for file source ---
while True:
    print("-" * 50)
    method = input(
        "How do you want to provide the package ZIP file?\n"
        "1. Upload directly to Colab.\n"
        "2. Use Google Drive.\n"
        "Enter choice (1 or 2): "
    ).strip()
    if method in ['1', '2']:
        break
    else:
        print("Invalid choice. Please enter 1 or 2.")
print("-" * 50)

# --- Handle file source ---
if method == '1':
    print("Selected: Upload directly to Colab.")
    try:
        uploaded = files.upload()
        if uploaded:
            uploaded_filename = list(uploaded.keys())[0]
            zip_filepath = os.path.join("/content", uploaded_filename)
            print(f"\nSuccessfully uploaded: '{uploaded_filename}'")
    except Exception as e:
        print(f"\nAn error occurred during upload: {e}")
elif method == '2':
    using_gdrive = True
    print("Selected: Use Google Drive.")
    try:
        drive.mount('/content/drive', force_remount=True)
        print("Google Drive mounted successfully.")
        if not os.path.isdir(GDRIVE_I2DC_PATH):
            print(f"\n❌ ERROR: The folder '{GDRIVE_I2DC_PATH}' does not exist.")
            print("Please create a folder named 'i2dc' in 'My Drive'.")
        else:
            zip_files_found = [f for f in os.listdir(GDRIVE_I2DC_PATH) if f.lower().endswith('.zip')]
            if len(zip_files_found) == 0:
                print(f"No .zip files found in '{GDRIVE_I2DC_PATH}'.")
            elif len(zip_files_found) == 1:
                zip_filename = zip_files_found[0]
                zip_filepath = os.path.join(GDRIVE_I2DC_PATH, zip_filename)
                print(f"Found unique ZIP file: '{zip_filename}'")
            else: # Handle multiple zips
                print("\nMultiple .zip files found. Please choose one:")
                for i, filename in enumerate(zip_files_found): print(f"  {i+1}. {filename}")
                choice = int(input(f"Enter the number (1-{len(zip_files_found)}): ")) - 1
                chosen_filename = zip_files_found[choice]
                zip_filepath = os.path.join(GDRIVE_I2DC_PATH, chosen_filename)
                print(f"You selected: '{chosen_filename}'")
    except Exception as e:
        print(f"\nAn error occurred with Google Drive: {e}")

# --- Unzip and Load Data ---
if zip_filepath and os.path.exists(zip_filepath):
    print(f"\n🔄 Unzipping {os.path.basename(zip_filepath)}...")
    with zipfile.ZipFile(zip_filepath, 'r') as z: z.extractall(REVIEW_DIR)
    print("✅ Unzip complete.")

    search_path = os.path.join(REVIEW_DIR, METADATA_FILENAME_PATTERN)
    metadata_files_found = glob.glob(search_path)
    if not metadata_files_found:
        print(f"\n❌ ERROR: Could not find a metadata file in the ZIP.")
    else:
        METADATA_FILE_PATH = metadata_files_found[0]
        print(f"\n📊 Loading session from: {os.path.basename(METADATA_FILE_PATH)}")
        df = pd.read_excel(METADATA_FILE_PATH)

        cols_to_process = ['additional_files', 'cover_image_url', 'title', 'keywords', 'abstract', 'fulltext_url']
        for col in cols_to_process:
            if col not in df.columns: df[col] = ''
        df.fillna('', inplace=True)
        for col in cols_to_process: df[col] = df[col].astype(str)

        df['_is_removed'] = False
        print("✅ New session started.")

        # --- NEW: Filter for video files only ---
        print("\n--- Filtering for Video Files ---")
        initial_count = len(df)
        video_extensions = ['.mp4', '.mov', '.webm']
        # Ensure 'fulltext_url' is string type before using .str accessor
        df['fulltext_url'] = df['fulltext_url'].astype(str)
        df = df[df['fulltext_url'].str.lower().str.endswith(tuple(video_extensions))].copy()
        df.reset_index(drop=True, inplace=True) # Reset index to be sequential
        filtered_count = len(df)

        print(f"✅ Kept {filtered_count} video records (MP4, MOV, WebM).")
        if initial_count > filtered_count:
            print(f"ℹ️  {initial_count - filtered_count} non-video records (e.g., images) were filtered out and will be ignored.")

    # --- Pre-process all videos to generate thumbnails ---
    if df is not None and not df.empty:
        print("\n--- Pre-processing: Generating Video Thumbnails ---")
        videos_to_process = []
        for index, row in df.iterrows():
            media_filename = str(row.get('fulltext_url', ''))
            # This check is now slightly redundant due to filtering, but it's good practice
            if any(media_filename.lower().endswith(ext) for ext in video_extensions):
                cover_image = str(row.get('cover_image_url', ''))
                if not cover_image or not os.path.exists(os.path.join(REVIEW_DIR, cover_image)):
                    videos_to_process.append((index, media_filename))

        if not videos_to_process:
            print("✅ No new video thumbnails needed.")
        else:
            total_videos = len(videos_to_process)
            for i, (index, media_filename) in enumerate(videos_to_process):
                print(f"  ({i+1}/{total_videos}) Generating thumbnail for '{media_filename}'...")
                video_filepath = os.path.join(REVIEW_DIR, media_filename)
                thumb_filename = f"{os.path.splitext(os.path.basename(media_filename))[0]}_thumb.jpg"
                thumb_path = os.path.join(REVIEW_DIR, thumb_filename)
                if os.path.exists(video_filepath) and generate_thumbnail(video_filepath, thumb_path):
                    df.loc[index, 'cover_image_url'] = thumb_filename
            print("✅ Thumbnail generation complete.")

        print("\n---> You are now ready to proceed to Step 3: Interactive Review.")
    elif df is not None and df.empty:
        print("\n❌ No video files were found in the package. Cannot proceed to review.")
    else:
        print("\n❌ Failed to load any data. Cannot proceed.")
else:
    print("\nNo valid ZIP file was provided. Cannot proceed.")

In [ ]:
#@title <h1> **Step 3: Interactive Triage & Review**
# @markdown Run this cell to review metadata, generate SRT files with whisper (using your CPU), and flag items for removal.

import ipywidgets as widgets
from IPython.display import display, clear_output, Image, Video
import pandas as pd
import time
import subprocess
import os

if 'df' not in locals() or df is None:
    print("❌ Metadata DataFrame is not loaded. Please successfully run Step 2 first.")
else:
    # --- Global state ---
    current_index = 0

    # --- UI Widget Definitions ---
    field_layout = widgets.Layout(width='95%')
    textarea_layout = widgets.Layout(width='95%', height='120px')
    srt_textarea_layout = widgets.Layout(width='95%', height='150px')

    title_widget = widgets.Text(description='Title:', layout=field_layout, style={'description_width': 'initial'})
    abstract_widget = widgets.Textarea(description='Post Text (in quotes):', layout=textarea_layout, style={'description_width': 'initial'})
    description_widget = widgets.Textarea(description='Description:', layout=textarea_layout, placeholder='Optional: Add a description here. It will be appended to the Post Text.', style={'description_width': 'initial'})
    keywords_widget = widgets.Text(description='Keywords:', layout=field_layout, placeholder='comma, separated, values', style={'description_width': 'initial'})

    # --- Add or Generate SRT Widgets (REVISED) ---
    whisper_model_dropdown = widgets.Dropdown(options=['tiny', 'base', 'small', 'medium'], value='tiny', description='Whisper Model:')
    generate_srt_button = widgets.Button(description='Generate SRT with AI (CPU)', icon='cogs', button_style='primary')

    # This textarea is now for pasting or for receiving AI-generated content
    add_edit_srt_textarea = widgets.Textarea(
        description='Add/Edit SRT Content:',
        placeholder='Paste SRT content here, or use the button above to generate it.',
        layout=srt_textarea_layout,
        style={'description_width': 'initial'},
        disabled=False # Always enabled to allow pasting
    )

    new_srt_filename_input = widgets.Text(description='Save As:', layout=widgets.Layout(width='70%'), style={'description_width': 'initial'})
    save_srt_button = widgets.Button(description='Save', icon='save', button_style='success', layout=widgets.Layout(width='auto'))

    # Save box is now always visible
    save_srt_box = widgets.HBox([new_srt_filename_input, save_srt_button], layout={'display': 'flex', 'align_items': 'center', 'margin': '5px 0'})

    add_srt_box = widgets.VBox([
        widgets.HTML("<b>Add or Generate SRT</b>"),
        widgets.HBox([whisper_model_dropdown, generate_srt_button]),
        add_edit_srt_textarea,
        save_srt_box
    ], layout={'border': '1px solid #cccccc', 'padding': '10px', 'margin': '5px 0'})


    # --- Active SRT Management Widgets ---
    srt_selector_dropdown = widgets.Dropdown(description='Active SRT:', layout=field_layout, style={'description_width': 'initial'})
    active_srt_editor = widgets.Textarea(description='SRT Content (view/edit):', layout=srt_textarea_layout, style={'description_width': 'initial'})
    remove_srt_button = widgets.Button(description='Remove Selected SRT', button_style='warning', icon='unlink')
    srt_management_box = widgets.VBox([
        widgets.HTML("<b>Active SRT Management</b>"),
        srt_selector_dropdown,
        active_srt_editor,
        remove_srt_button
    ], layout={'border': '1px solid #cccccc', 'padding': '10px', 'margin': '5px 0'})

    # --- Main Layout and Control Widgets ---
    session_status_label = widgets.Label(value="")
    prev_button, next_button = widgets.Button(description='< Previous'), widgets.Button(description='Next >')
    remove_button, restore_button = widgets.Button(description='Remove this Item', button_style='danger', icon='trash'), widgets.Button(description='Restore this Item', button_style='success', icon='undo', layout={'display': 'none'})
    progress_label, goto_input, goto_button = widgets.Label(), widgets.IntText(description='Go to:', value=1), widgets.Button(description='Go')
    media_output = widgets.Output()
    load_full_video_button = widgets.Button(description='Load Full Video', icon='play', button_style='primary', layout={'display':'flex'})
    media_area = widgets.VBox([media_output, load_full_video_button], layout={'align_items': 'center', 'width': '420px'})

    # --- Helper & Logic Functions ---
    def find_next_valid_index(start_index):
        valid_indices = df[~df['_is_removed']].index
        return valid_indices[valid_indices > start_index][0] if any(valid_indices > start_index) else valid_indices[0] if any(valid_indices) else None

    def find_prev_valid_index(start_index):
        valid_indices = df[~df['_is_removed']].index
        return valid_indices[valid_indices < start_index][-1] if any(valid_indices < start_index) else valid_indices[-1] if any(valid_indices) else None

    def save_current_record():
        if current_index is None or df.loc[current_index, '_is_removed']:
            return

        df.loc[current_index, 'title'] = title_widget.value
        df.loc[current_index, 'keywords'] = keywords_widget.value

        post_text = abstract_widget.value.strip().replace('"', '"')
        post_description = description_widget.value.strip()

        final_abstract = f'<b>Posted Text: </b><br>{post_text}'
        if post_description:
            final_abstract += f'<br><br><b>Description: </b><br>{post_description}'
        df.loc[current_index, 'abstract'] = final_abstract

        selected_srt = srt_selector_dropdown.value
        if selected_srt and not active_srt_editor.disabled:
            try:
                srt_path = os.path.join(REVIEW_DIR, selected_srt)
                if os.path.exists(srt_path):
                    with open(srt_path, 'w', encoding='utf-8') as f:
                        f.write(active_srt_editor.value)
            except Exception as e:
                session_status_label.value = f"🔥 Error saving SRT: {e}"


    def display_record(index):
        global current_index
        current_index = index
        if current_index is None:
            with media_output: clear_output(wait=True); print("No items to review.")
            progress_label.value = "No items to review"
            return

        record = df.loc[current_index]
        is_removed = bool(record['_is_removed'])
        total_rem = (~df['_is_removed']).sum()
        progress_label.value = f'Item: {current_index + 1} of {len(df)} (Remaining: {total_rem})'
        goto_input.value = current_index + 1
        session_status_label.value = ""

        # Enable/Disable widgets based on removed status
        all_editors = [title_widget, abstract_widget, description_widget, keywords_widget, generate_srt_button, active_srt_editor, remove_srt_button, srt_selector_dropdown, add_edit_srt_textarea, new_srt_filename_input, save_srt_button]
        for w in all_editors: w.disabled = is_removed
        remove_button.layout.display, restore_button.layout.display = ('none', 'flex') if is_removed else ('flex', 'none')

        # Load metadata
        title_widget.value = record.get('title', '')
        keywords_widget.value = record.get('keywords', '')
        abstract_widget.value = ''; description_widget.value = ''

        raw_abstract = str(record.get('abstract', ''))
        parts = raw_abstract.split('<br><br><b>Description: </b><br>')
        post_text_part = parts[0].replace('<b>Posted Text: </b><br>', '').replace('"', '"')
        abstract_widget.value = post_text_part
        if len(parts) > 1:
            description_widget.value = parts[1]

        # Reset and configure SRT and Media widgets
        media_filename = record.get('fulltext_url', '')
        is_video = any(media_filename.lower().endswith(ext) for ext in ['.mp4', '.mov', '.webm'])
        add_srt_box.layout.display = 'flex' if is_video else 'none'
        srt_management_box.layout.display = 'flex' if is_video else 'none'
        add_edit_srt_textarea.value = "" # Clear paste box

        if is_video:
            base_name = os.path.splitext(os.path.basename(media_filename))[0]
            new_srt_filename_input.value = f"{base_name}_en.srt" # Suggest a default filename
            update_srt_widgets() # Populate SRT dropdown and editor
        else:
            active_srt_editor.value = ""

        with media_output:
            clear_output(wait=True)
            if not media_filename: print("No media file specified for this record.")
            else:
                media_filepath = os.path.join(REVIEW_DIR, media_filename)
                is_image = any(media_filename.lower().endswith(ext) for ext in ['.jpg', '.jpeg', '.png', '.gif', '.webp'])
                if os.path.exists(media_filepath):
                    if is_image:
                        load_full_video_button.layout.display = 'none'
                        display(Image(filename=media_filepath, width=400))
                    elif is_video:
                        load_full_video_button.layout.display = 'flex'
                        cover_image_path = os.path.join(REVIEW_DIR, record.get('cover_image_url', ''))
                        if record.get('cover_image_url') and os.path.exists(cover_image_path):
                            display(Image(filename=cover_image_path, width=400))
                        else: display(Image(filename=VIDEO_PLACEHOLDER_PATH))
                    else:
                        load_full_video_button.layout.display = 'none'; print(f"Unsupported file type: {media_filename}")
                else:
                    load_full_video_button.layout.display = 'none'
                    print(f"❌ Media file not found: {media_filepath}")


    def update_srt_widgets():
        record = df.loc[current_index]
        is_removed = bool(record['_is_removed'])
        srt_filenames = sorted([f.strip() for f in str(record.get('additional_files', '')).split('|') if f.strip().lower().endswith('.srt')])

        active_srt_editor.value = ""
        srt_selector_dropdown.options = srt_filenames
        if srt_filenames:
            srt_selector_dropdown.disabled = is_removed
            active_srt_editor.disabled = is_removed
            remove_srt_button.disabled = is_removed
            if srt_selector_dropdown.value not in srt_filenames:
                srt_selector_dropdown.value = srt_filenames[0]
            else:
                load_srt_content(srt_selector_dropdown.value)
        else:
            srt_selector_dropdown.disabled = True
            active_srt_editor.disabled = True
            remove_srt_button.disabled = True
            active_srt_editor.value = "--- No SRT file associated with this record. ---"

    def load_srt_content(filename):
        if not filename: return
        srt_path = os.path.join(REVIEW_DIR, filename)
        if os.path.exists(srt_path):
            with open(srt_path, 'r', encoding='utf-8', errors='ignore') as f:
                active_srt_editor.value = f.read()
        else:
            active_srt_editor.value = f"--- SRT FILE '{filename}' NOT FOUND ---\n"

    def on_srt_selection_changed(change):
        if change['type'] == 'change' and change['name'] == 'value':
            load_srt_content(change['new'])

    def on_load_full_video_clicked(b):
        with media_output:
            clear_output(wait=True); record = df.loc[current_index]
            video_path = os.path.join(REVIEW_DIR, record.get('fulltext_url', ''))
            if os.path.exists(video_path): display(Video(video_path, width=400, embed=True)); load_full_video_button.layout.display = 'none'
            else: print(f"❌ Video file not found: {video_path}")

    def on_nav_clicked(b): save_current_record(); display_record(find_next_valid_index(current_index) if b.description == 'Next >' else find_prev_valid_index(current_index))
    def on_goto_clicked(b): save_current_record(); display_record(max(0, min(len(df) - 1, goto_input.value - 1)))
    def on_remove_restore_clicked(b):
        is_restoring = (b.description == 'Restore this Item'); df.loc[current_index, '_is_removed'] = not is_restoring
        display_record(current_index)
        if not is_restoring:
            next_idx = find_next_valid_index(current_index)
            if next_idx is not None:
                display_record(next_idx)

    def on_remove_srt_clicked(b):
        selected_srt = srt_selector_dropdown.value
        if not selected_srt:
            session_status_label.value = "⚠️ No SRT selected to remove."
            return
        all_files = [f.strip() for f in str(df.loc[current_index, 'additional_files']).split('|') if f.strip() and f.strip() != selected_srt]
        df.loc[current_index, 'additional_files'] = '|'.join(all_files)
        session_status_label.value = f"✅ '{selected_srt}' removed from record."
        update_srt_widgets()

    def on_generate_srt_clicked(b):
        b.disabled = True
        session_status_label.value = "Starting AI SRT Generation..."
        video_filename = df.loc[current_index].get('fulltext_url', '')
        video_filepath = os.path.join(REVIEW_DIR, video_filename)
        if not video_filename or not os.path.exists(video_filepath):
            session_status_label.value = f"❌ Video file not found: {video_filepath}"; b.disabled = False; return

        temp_audio_path = "/content/temp_audio.mp3"
        base_name_for_temp = os.path.splitext(os.path.basename(video_filename))[0]
        generated_srt_temp_path = f"/content/{base_name_for_temp}.srt"
        # Ensure the temporary file is removed if it exists from a previous run
        if os.path.exists(generated_srt_temp_path): os.remove(generated_srt_temp_path)


        try:
            session_status_label.value = "🔄 Extracting audio from video..."
            audio_command = ['ffmpeg', '-i', video_filepath, '-y', '-vn', '-q:a', '0', '-map', 'a', temp_audio_path]
            subprocess.run(audio_command, check=True, capture_output=True, text=True)

            model = whisper_model_dropdown.value
            session_status_label.value = f"🤖 Transcribing with Whisper ('{model}' on CPU)... This may take a while."
            whisper_command = ['whisper', temp_audio_path, '--model', model, '--language', 'en', '--output_format', 'srt', '--output_dir', '/content', '--device', 'cpu']
            print(f"Executing Whisper command: {' '.join(whisper_command)}") # Debug print
            result = subprocess.run(whisper_command, check=True, capture_output=True, text=True)
            print(f"Whisper STDOUT:\n{result.stdout}") # Debug print
            print(f"Whisper STDERR:\n{result.stderr}") # Debug print


            # --- Modification Start ---
            # Wait briefly to ensure file system sync
            time.sleep(1)
            if os.path.exists(generated_srt_temp_path):
                 with open(generated_srt_temp_path, 'r', encoding='utf-8', errors='ignore') as f:
                    srt_content = f.read()
                 add_edit_srt_textarea.value = srt_content
                 session_status_label.value = f"✅ AI SRT generated and loaded. Review content and save."
            else:
                 raise FileNotFoundError("Whisper did not produce an SRT file at the expected location.")
            # --- Modification End ---


        except subprocess.CalledProcessError as e:
            session_status_label.value = f"❌ ERROR during transcription. Check runtime logs."
            print(f"--- FFMPEG/WHISPER STDERR ---\n{e.stderr}\n--------------------------")
        except Exception as e:
            session_status_label.value = f"❌ An error occurred: {e}"
        finally:
            if os.path.exists(temp_audio_path): os.remove(temp_audio_path)
            # Keep the generated SRT file in /content for inspection if needed, but remove it at the start of the function
            # if os.path.exists(generated_srt_temp_path): os.remove(generated_srt_temp_path)
            b.disabled = False


    def on_save_srt_clicked(b):
        b.disabled = True
        try:
            final_srt_filename = new_srt_filename_input.value.strip()
            if not final_srt_filename:
                session_status_label.value = "❌ Filename cannot be empty."; b.disabled=False; return
            if not final_srt_filename.lower().endswith('.srt'):
                final_srt_filename += '.srt'

            final_srt_filepath = os.path.join(REVIEW_DIR, final_srt_filename)
            session_status_label.value = "Saving..."

            with open(final_srt_filepath, 'w', encoding='utf-8') as f:
                f.write(add_edit_srt_textarea.value)

            current_files_str = str(df.loc[current_index, 'additional_files'])
            all_files = [f.strip() for f in current_files_str.split('|') if f.strip()]
            if final_srt_filename not in all_files:
                all_files.append(final_srt_filename)
            df.loc[current_index, 'additional_files'] = '|'.join(sorted(all_files))

            session_status_label.value = f" ✅ Saved as '{final_srt_filename}' and linked to record."
            add_edit_srt_textarea.value = ''

            update_srt_widgets()
            srt_selector_dropdown.value = final_srt_filename

        except Exception as e:
            session_status_label.value = f"❌ Error saving generated SRT: {e}"
        finally:
             b.disabled = False


    # --- Link Handlers ---
    next_button.on_click(on_nav_clicked); prev_button.on_click(on_nav_clicked); goto_button.on_click(on_goto_clicked)
    remove_button.on_click(on_remove_restore_clicked); restore_button.on_click(on_remove_restore_clicked)
    load_full_video_button.on_click(on_load_full_video_clicked)
    generate_srt_button.on_click(on_generate_srt_clicked)
    save_srt_button.on_click(on_save_srt_clicked)
    remove_srt_button.on_click(on_remove_srt_clicked)
    srt_selector_dropdown.observe(on_srt_selection_changed, names='value')


    # --- Assemble UI ---
    nav_controls = widgets.HBox([prev_button, progress_label, next_button], layout=widgets.Layout(justify_content='space-around'))
    goto_controls = widgets.HBox([goto_input, goto_button])
    top_controls = widgets.HBox([nav_controls, goto_controls, remove_button, restore_button], layout=widgets.Layout(justify_content='space-between', align_items='center'))
    metadata_editors = widgets.VBox([title_widget, abstract_widget, description_widget, keywords_widget])
    right_panel = widgets.VBox([metadata_editors, add_srt_box, srt_management_box, session_status_label])
    main_review_area = widgets.HBox([media_area, right_panel], layout=widgets.Layout(align_items='flex-start'))
    reviewer_ui = widgets.VBox([top_controls, main_review_area])

    # --- Initial display ---
    display(reviewer_ui)
    display_record(find_next_valid_index(-1))

In [ ]:
#@title <h1> **Step 4: Finalize and Download Package**

# @markdown Run this cell to save your work, create a final package, and then either download it or save it back to Google Drive.

import pandas as pd
from google.colab import files
import os
import shutil
import zipfile

def create_final_package(df_to_package, staging_dir, zip_base_name):
    """Helper function to create a zip package from a dataframe."""
    if os.path.exists(staging_dir): shutil.rmtree(staging_dir)
    os.makedirs(staging_dir)

    # Copy all necessary media files to the staging area
    for index, row in df_to_package.iterrows():
        for col in ['fulltext_url', 'additional_files', 'cover_image_url']:
             if pd.notna(row[col]) and row[col]:
                for filename in str(row[col]).split('|'):
                    filename = filename.strip()
                    if not filename: continue
                    source_path = os.path.join(REVIEW_DIR, filename)
                    if os.path.exists(source_path):
                        shutil.copy2(source_path, os.path.join(staging_dir, os.path.basename(filename)))
                    else:
                        print(f"  ⚠️ WARNING: File '{filename}' not found. It will not be included in the package.")

    # Remove the temporary '_is_removed' column and save the final metadata file
    final_df = df_to_package.drop(columns=['_is_removed'], errors='ignore')
    final_df.to_excel(os.path.join(staging_dir, "metadata.xlsx"), index=False, engine='openpyxl')

    # Create the final ZIP archive
    zip_path = shutil.make_archive(zip_base_name, 'zip', staging_dir)
    return zip_path

# --- Main Execution Logic ---
if 'df' not in locals() or df is None:
    print("❌ DataFrame not available. Cannot save. Please run Steps 1, 2 and 3 first.")
else:
    # Ensure the very last edits are saved before packaging
    if 'current_index' in locals() and current_index is not None:
        save_current_record()
        print("Final check: All changes from the current view have been captured.")
    else:
        print("Starting finalization process.")

    # Create a clean DataFrame with only the records that were not removed
    df_final = df[~df['_is_removed']].copy()

    print("\n" + "-" * 50)
    print("Finalization Summary:")
    print(f"  - {len(df_final)} items will be included in the final package.")
    print(f"  - {df['_is_removed'].sum()} items were removed and will be excluded.")
    print("-" * 50)

    # Determine the base name for the output file
    base_name = os.path.splitext(os.path.basename(zip_filepath))[0] if 'zip_filepath' in locals() and zip_filepath else "reviewed_package"

    if not df_final.empty:
        print("\n📦 Creating the final package...")
        final_staging_dir = "/content/final_package"
        final_zip_base_name = f"/content/{base_name}_REVIEWED"

        ready_zip_path = create_final_package(df_final, final_staging_dir, final_zip_base_name)
        print(f"✅ Successfully created '{os.path.basename(ready_zip_path)}'.")

        # --- Ask for save/download destination ---
        save_to_gdrive = False
        # Check if Google Drive was used in Step 2 and the path still exists
        if 'using_gdrive' in locals() and using_gdrive and 'GDRIVE_I2DC_PATH' in locals() and os.path.isdir(GDRIVE_I2DC_PATH):
            while True:
                print("\n" + "-" * 50)
                save_method = input(
                    "Where would you like to save the final package?\n"
                    "1. Download directly to your computer.\n"
                    f"2. Save to your Google Drive ('{GDRIVE_I2DC_PATH}').\n"
                    "Enter choice (1 or 2): "
                ).strip()
                if save_method in ['1', '2']:
                    if save_method == '2':
                        save_to_gdrive = True
                    break
                else:
                    print("Invalid choice. Please enter 1 or 2.")
            print("-" * 50)

        # --- Execute the chosen save/download method ---
        if save_to_gdrive:
            try:
                dest_path = os.path.join(GDRIVE_I2DC_PATH, os.path.basename(ready_zip_path))
                print(f"\n📤 Saving file to Google Drive at: {dest_path}...")
                shutil.move(ready_zip_path, dest_path)
                print(f"✅ Successfully saved package to your Google Drive.")
            except Exception as e:
                print(f"\n❌ An error occurred while saving to Google Drive: {e}")
                print("Falling back to direct download...")
                files.download(ready_zip_path)
        else:
            print("\n⬇️ Preparing download...")
            files.download(ready_zip_path)

    else:
        print("\nℹ️ No items remained after review. Skipping final package creation.")

    # --- New instruction block ---
    print("\n" + "="*60)
    print("✅ PROCESS COMPLETE.")
    print("\nTo process another ZIP file, you can now scroll up and run 'Step 2: Load ZIP Package' again.")
    print("You do NOT need to run Step 1 again in this session.")
    print("="*60)